In [15]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('marketing.db')
print("Database Connected ✅")

Database Connected ✅


In [19]:
ad_spend = pd.read_csv(r'C:\Users\HP\Desktop\Project 1 - Marketing attribution\Data\Processed\cleaned_ad_spend.csv')

web_log = pd.read_csv(r'C:\Users\HP\Desktop\Project 1 - Marketing attribution\Data\Processed\cleaned_web_log.csv')

crm = pd.read_csv(r'C:\Users\HP\Desktop\Project 1 - Marketing attribution\Data\Processed\cleaned_crm.csv')

ad_spend.to_sql('ad_spend', conn, if_exists='replace', index=False)
web_log.to_sql('web_log', conn, if_exists='replace', index=False)
crm.to_sql('crm', conn, if_exists='replace', index=False)

print("Tables Created ✅")

Tables Created ✅


In [21]:
query = """
SELECT 
    user_id,
    channel,
    timestamp,
    ROW_NUMBER() OVER (
        PARTITION BY user_id 
        ORDER BY timestamp ASC
    ) AS touchpoint_number,
    COUNT(*) OVER (
        PARTITION BY user_id
    ) AS total_touchpoints
FROM web_log
"""

journey = pd.read_sql_query(query, conn)

print(journey.head(10))
print("User Journey Created ✅")

     user_id        channel                  timestamp  touchpoint_number  \
0  USR_00001  Meta_Facebook  2024-03-07 21:21:00+00:00                  1   
1  USR_00002  Meta_Facebook  2024-03-25 00:58:00+00:00                  1   
2  USR_00002        Organic  2024-03-27 20:39:00+00:00                  2   
3  USR_00003  Google_Search  2024-03-06 08:26:00+00:00                  1   
4  USR_00003         TikTok  2024-03-06 13:53:00+00:00                  2   
5  USR_00003  Meta_Facebook  2024-03-12 20:46:00+00:00                  3   
6  USR_00003         TikTok  2024-03-12 22:21:00+00:00                  4   
7  USR_00003  Meta_Facebook  2024-03-13 02:06:00+00:00                  5   
8  USR_00004  Meta_Facebook  2024-01-07 19:05:00+00:00                  1   
9  USR_00004  Google_Search  2024-01-08 02:19:00+00:00                  2   

   total_touchpoints  
0                  1  
1                  2  
2                  2  
3                  5  
4                  5  
5             

In [23]:
conn.execute("DROP VIEW IF EXISTS journey_view")

conn.execute("""
CREATE VIEW journey_view AS
SELECT
    user_id,
    channel,
    timestamp,
    ROW_NUMBER() OVER (
        PARTITION BY user_id
        ORDER BY timestamp ASC
    ) AS touchpoint_number
FROM web_log
""")

first_touch = pd.read_sql_query("""
SELECT
    j.channel,
    SUM(c.revenue_usd) AS attributed_revenue,
    COUNT(*) AS conversions
FROM journey_view j
JOIN crm c
ON j.user_id = c.customer_id
WHERE j.touchpoint_number = 1
GROUP BY j.channel
ORDER BY attributed_revenue DESC
""", conn)

print("=== FIRST TOUCH ATTRIBUTION ===")
print(first_touch)

=== FIRST TOUCH ATTRIBUTION ===
         channel  attributed_revenue  conversions
0  Meta_Facebook            42265.55          154
1  Google_Search            41979.91          158
2         TikTok            27134.50          100
3          Email            16059.94           68
4       LinkedIn            15520.59           55
5        Organic            14518.78           65


In [25]:
last_touch = pd.read_sql_query("""
SELECT 
    last_touch_channel AS channel,
    SUM(revenue_usd) AS attributed_revenue,
    COUNT(*) AS conversions
FROM crm
GROUP BY last_touch_channel
ORDER BY attributed_revenue DESC
""", conn)

print("=== LAST TOUCH ATTRIBUTION ===")
print(last_touch)

=== LAST TOUCH ATTRIBUTION ===
         channel  attributed_revenue  conversions
0  Google_Search            43583.44          167
1  Meta_Facebook            41564.63          149
2         TikTok            24587.92          102
3       LinkedIn            16716.83           61
4        Organic            16696.59           64
5          Email            14329.86           57


In [27]:
linear = pd.read_sql_query("""
SELECT
    j.channel,
    SUM(c.revenue_usd * 1.0 / j.total_touchpoints) AS attributed_revenue,
    COUNT(DISTINCT j.user_id) AS users
FROM (
    SELECT
        user_id,
        channel,
        COUNT(*) OVER (PARTITION BY user_id) AS total_touchpoints
    FROM web_log
) j
JOIN crm c
ON j.user_id = c.customer_id
GROUP BY j.channel
ORDER BY attributed_revenue DESC
""", conn)

print("=== LINEAR ATTRIBUTION ===")
print(linear)

=== LINEAR ATTRIBUTION ===
         channel  attributed_revenue  users
0  Google_Search        44163.159833    356
1  Meta_Facebook        40732.084833    339
2         TikTok        25506.307167    236
3       LinkedIn        17036.847167    178
4        Organic        15593.794667    166
5          Email        14447.076333    152


In [29]:
first_touch.to_csv(
    r'C:\Users\HP\Desktop\Project 1 - Marketing attribution\Data\Processed\first_touch_attribution.csv',
    index=False
)

last_touch.to_csv(
    r'C:\Users\HP\Desktop\Project 1 - Marketing attribution\Data\Processed\last_touch_attribution.csv',
    index=False
)

linear.to_csv(
    r'C:\Users\HP\Desktop\Project 1 - Marketing attribution\Data\Processed\linear_attribution.csv',
    index=False
)

print("Attribution Files Saved ✅")

Attribution Files Saved ✅


In [31]:
first_touch.to_csv('first_touch_attribution.csv', index=False)
last_touch.to_csv('last_touch_attribution.csv', index=False)
linear.to_csv('linear_attribution.csv', index=False)

print("Attribution Files Saved ✅")

Attribution Files Saved ✅
